# Fake News Detection — TF-IDF Baseline

Building a binary classifier on a small synthetic fake-vs-real news corpus (600 rows). I tried pulling the LIAR dataset via HuggingFace `datasets` first but the auto-download was failing on my machine, so I fell back to a synthetic stand-in. The class structure mirrors LIAR's pants-on-fire vs true split — see notes at the bottom.

The honest version of this problem is harder than the binary framing makes it look. Real-world "fake news" is mostly *partially* true, framed misleadingly, or just emphasizes some facts over others. A binary label flattens all of that, which is worth remembering before drawing any product conclusions from these results.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(0)
DATA_DIR = Path("../../data/fake_news")


In [2]:
df = pd.read_csv(DATA_DIR / "fake_news.csv")
print(df.shape, "  fake rate:", df.label.mean())
df.head()


(600, 2)   fake rate: 0.5


,text,label
0,India signs trade agreement with Canada on ant...,0
1,Local mom discovers tap water reverses aging —...,1
2,Whistleblower reveals a senator planned to min...,1
3,Germany central bank raises interest rates by ...,0
4,Government secretly stockpiling 5G towers for ...,1


## TF-IDF + LogReg, with a Naive Bayes comparison

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"].values, df["label"].values,
    test_size=0.25, stratify=df["label"], random_state=0
)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

results = {}
for name, clf in [("LogReg", LogisticRegression(max_iter=2000, C=1.0)),
                  ("MultinomialNB", MultinomialNB())]:
    clf.fit(Xtr, y_train)
    proba = clf.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results[name] = {
        "auc": roc_auc_score(y_test, proba),
    }
    print(f"--- {name} ---")
    print(classification_report(y_test, pred, target_names=["real", "fake"]))

results


--- LogReg ---
              precision    recall  f1-score   support

        real       1.00      1.00      1.00        75
        fake       1.00      1.00      1.00        75

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150

--- MultinomialNB ---
              precision    recall  f1-score   support

        real       1.00      1.00      1.00        75
        fake       1.00      1.00      1.00        75

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



{'LogReg': {'auc': 1.0}, 'MultinomialNB': {'auc': 1.0}}

## Notes / honesty section

What this notebook actually shows: TF-IDF can separate two text distributions when they were generated from genuinely different templates. That is what's happening here — my synthetic "fake" and "real" templates use different vocabulary patterns (BREAKING / SHOCKING / EXPOSED on one side; central bank / parliament / quarterly on the other) so the classifier basically learns "headline-ish caps and clickbait phrasing = fake."

On the real LIAR dataset (or PolitiFact / fakenewsnet), the gap is much smaller because real and fake news share most of the vocabulary. The published TF-IDF + LR baselines on LIAR are in the 0.25-0.30 macro-F1 range (LIAR is six-class, not binary), and even binarized accuracy hovers around 60% — well below what you'd get here.

For a serious version of this:
- Try the LIAR dataset properly (six-way classification + speaker metadata features).
- Add source/URL features. They're often more predictive than the text.
- Don't deploy a binary classifier as the final arbiter. It's a triage tool at best.
